In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURACIÓN VISUAL
# ─────────────────────────────────────────────────────────────────────────────
PALETTE   = ["#1B4F72", "#2E86C1", "#85C1E9", "#F4D03F", "#E74C3C"]
PALETTE_5 = ["#E74C3C", "#E67E22", "#F4D03F", "#2ECC71", "#1B4F72"]
sns.set_theme(style="whitegrid", palette=PALETTE)
plt.rcParams.update({
    "figure.facecolor": "#F8F9FA",
    "axes.facecolor"  : "#FFFFFF",
    "axes.spines.top" : False,
    "axes.spines.right": False,
    "font.family"     : "DejaVu Sans",
    "axes.titlesize"  : 13,
    "axes.labelsize"  : 11,
})

In [14]:
import sys
sys.path.append("..")   # para que encuentre src/

from src.utils import load_raw_csvs, build_master_table, save_master_table

dfs = load_raw_csvs()
print({k: v.shape for k, v in dfs.items()})

master = build_master_table(dfs)
save_master_table(master)

master.head()

{'customers': (99441, 5), 'geolocation': (1000163, 5), 'order_items': (112650, 7), 'payments': (103886, 5), 'reviews': (99224, 7), 'orders': (99441, 8), 'products': (32951, 9), 'sellers': (3095, 4), 'category': (71, 2)}
Master Table guardada: 99,441 filas x 23 columnas


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,customer_zip_code_prefix,...,order_item_count,total_freight_value,total_price,seller_id,product_id,product_category_name_english,product_weight_g,product_photos_qty,review_score,review_comment_message
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,SP,3149,...,1.0,8.72,29.99,3504c0cb71d7fa48d967e0e4c94d59d9,87285b34884572647811a353c7ac498a,housewares,500.0,4.0,4.0,"Não testei o produto ainda, mas ele veio corre..."
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,BA,47813,...,1.0,22.76,118.70,289cdb325fb7e7f891c38608bf9e0962,595fac2a385ac33a80bd5114aec74eb8,perfumery,400.0,1.0,4.0,Muito bom o produto.
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,GO,75265,...,1.0,19.22,159.90,4869f7a5dfa277a7dca6462dcf3b52b2,aa4383b373c6aca5d8797843e5594415,auto,420.0,1.0,5.0,NaN
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,RN,59296,...,1.0,27.20,45.00,66922902710d126a0e7d26b0e3805106,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop,450.0,3.0,5.0,O produto foi exatamente o que eu esperava e e...
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,SP,9195,...,1.0,8.72,19.90,2c9e548be18521d1c43cde1c582c6de8,65266b2da20d04dbe00c5c2d3bb7859e,stationery,250.0,4.0,5.0,NaN


In [15]:
df = pd.read_csv("../data/processed/master_table.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order_id                       99441 non-null  object 
 1   customer_id                    99441 non-null  object 
 2   order_status                   99441 non-null  object 
 3   order_purchase_timestamp       99441 non-null  object 
 4   order_approved_at              99281 non-null  object 
 5   order_delivered_carrier_date   97658 non-null  object 
 6   order_delivered_customer_date  96476 non-null  object 
 7   order_estimated_delivery_date  99441 non-null  object 
 8   customer_state                 99441 non-null  object 
 9   customer_zip_code_prefix       99441 non-null  int64  
 10  payment_installments           99440 non-null  float64
 11  payment_value                  99440 non-null  float64
 12  payment_type                   99440 non-null 

In [16]:
# ═══════════════════════════════════════════════
# 1. CONVERSIÓN DE TIPOS (¡ESTO ES LO QUE FALTABA!)
# ═══════════════════════════════════════════════

# Fechas: de texto (object) a datetime
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Enteros: de float a int (los que no deben tener decimales)
int_cols = [
    "payment_installments",
    "order_item_count",
    "product_photos_qty",
    "review_score",
]
for col in int_cols:
    df[col] = df[col].fillna(0).astype(int)

# Códigos postales: de int a string (no se suman, son categorías)
df["customer_zip_code_prefix"] = df["customer_zip_code_prefix"].astype(str)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
 8   customer_state                 99441 non-null  object        
 9   customer_zip_code_prefix       99441 non-null  object        
 10  payment_installments           99441 non-null  int32         
 11  payment_value  

In [ ]:
# ═══════════════════════════════════════════════
# 2. FEATURE ENGINEERING
# ═══════════════════════════════════════════════

# ── Tiempos de entrega ──
df["dias_entrega"] = (
    df["order_delivered_customer_date"] - 
    df["order_purchase_timestamp"]
).dt.days

df["dias_retraso"] = (
    df["order_delivered_customer_date"] - 
    df["order_estimated_delivery_date"]
).dt.days

df["es_tarde"] = (df["dias_retraso"] > 0).astype(int)

# ── Fechas desglosadas ──
df["mes_compra"] = df["order_purchase_timestamp"].dt.to_period("M")
df["año_compra"] = df["order_purchase_timestamp"].dt.year

# ── Target y comentarios ──
df["satisfecho"] = (df["review_score"] >= 4).astype(int)
df["tiene_comentario"] = df["review_comment_message"].notna().astype(int)

# ── Valores del pedido ──
df["precio_promedio_item"] = df["total_price"] / (df["order_item_count"] + 1e-9)

# ── Pago en cuotas ──
df["pago_en_cuotas"] = (df["payment_installments"] > 1).astype(int)

# ── Nombres consistentes (tu master usa nombres diferentes) ──
# Renombrar para que coincidan con lo que pediste
df = df.rename(columns={
    "order_item_count": "total_items",
    "total_freight_value": "total_freight",
    "product_category_name_english": "product_category_name",
})

# ── Volumen del producto ──
# ⚠️ ESTAS COLUMNAS NO ESTÁN EN TU MASTER TABLE
# Si las tienes en otra tabla, debes hacer merge primero
# Por ahora las dejo comentadas:
# df["volumen_producto"] = (
#     df["product_length_cm"] * 
#     df["product_height_cm"] * 
#     df["product_width_cm"]
# )   

In [17]:
# ───────────────────────────────────────────────
# FEATURE ENGINEERING (nuevas columnas)
# ───────────────────────────────────────────────

# 1. Días de retraso en entrega (negativo = antes de tiempo, positivo = tarde)
df["delivery_delay_days"] = (
    df["order_delivered_customer_date"] - 
    df["order_estimated_delivery_date"]
).dt.days

# 2. Tiempo real de entrega desde compra hasta entrega
df["actual_delivery_days"] = (
    df["order_delivered_customer_date"] - 
    df["order_purchase_timestamp"]
).dt.days

# 3. Tiempo de aprobación del pago (en horas)
df["approval_time_hours"] = (
    df["order_approved_at"] - df["order_purchase_timestamp"]
).dt.total_seconds() / 3600

# 4. Porcentaje que representa el flete del total pagado
df["freight_ratio"] = df["total_freight_value"] / (
    df["total_price"] + df["total_freight_value"] + 1e-9
)

# 5. ¿El cliente dejó comentario? (1 = sí, 0 = no)
df["has_comment"] = df["review_comment_message"].notna().astype(int)

# 6. ¿Llegó a tiempo o antes? (1 = sí, 0 = tarde)
df["delivered_on_time"] = (df["delivery_delay_days"] <= 0).astype(int)

# 7. ¿Cliente satisfecho? (1 = score 4-5, 0 = score 1-3)
df["is_satisfied"] = (df["review_score"] >= 4).astype(int)

# 8. Mes de compra (formato año-mes)
df["purchase_month"] = df["order_purchase_timestamp"].dt.to_period("M")

# ───────────────────────────────────────────────
# FILTRAR SOLO PEDIDOS ENTREGADOS (para análisis)
# ───────────────────────────────────────────────
#df_delivered = df[df["order_status"] == "delivered"].copy()
    

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 31 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
 8   customer_state                 99441 non-null  object        
 9   customer_zip_code_prefix       99441 non-null  object        
 10  payment_installments           99441 non-null  int32         
 11  payment_value  

In [21]:
# ───────────────────────────────────────────────
# 1. TIEMPOS DE ENTREGA
# ───────────────────────────────────────────────

# Días de entrega real (desde compra hasta entrega al cliente)
df["dias_entrega"] = (
    df["order_delivered_customer_date"] - 
    df["order_purchase_timestamp"]
).dt.days

# Días de retraso (negativo = antes de tiempo, positivo = tarde)
df["dias_retraso"] = (
    df["order_delivered_customer_date"] - 
    df["order_estimated_delivery_date"]
).dt.days

# ¿Llegó tarde? (1 = sí, 0 = no)
df["es_tarde"] = (df["dias_retraso"] > 0).astype(int)

# ───────────────────────────────────────────────
# 2. FECHAS DESGLOSADAS
# ───────────────────────────────────────────────

# Mes de compra (como período año-mes)
df["mes_compra"] = df["order_purchase_timestamp"].dt.to_period("M")

# Año de compra
df["año_compra"] = df["order_purchase_timestamp"].dt.year

# ───────────────────────────────────────────────
# 3. TARGET Y COMENTARIOS
# ───────────────────────────────────────────────

# TARGET: ¿Cliente satisfecho? (1 = score 4-5, 0 = score 1-3)
df["satisfecho"] = (df["review_score"] >= 4).astype(int)

# ¿Dejó comentario escrito?
df["tiene_comentario"] = df["review_comment_message"].notna().astype(int)

# ───────────────────────────────────────────────
# 4. VALORES DEL PEDIDO
# ───────────────────────────────────────────────

# Precio promedio por item (evita división por cero)
#df["precio_promedio_item"] = df["total_price"] / (df["total_items"] + 1e-9)

# ───────────────────────────────────────────────
# 5. PAGO EN CUOTAS
# ───────────────────────────────────────────────

# ¿Pagó en cuotas? (1 = sí, más de 1 cuota; 0 = no, contado)
df["pago_en_cuotas"] = (df["payment_installments"] > 1).astype(int)

# ───────────────────────────────────────────────
# 6. VOLUMEN DEL PRODUCTO
# ───────────────────────────────────────────────

# Volumen = largo × alto × ancho (en cm³)
#df["volumen_producto"] = (
#    df["product_length_cm"] * 
#    df["product_height_cm"] * 
#    df["product_width_cm"]
#)

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 39 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
 8   customer_state                 99441 non-null  object        
 9   customer_zip_code_prefix       99441 non-null  object        
 10  payment_installments           99441 non-null  int32         
 11  payment_value  

In [24]:
# EDA - Análisis de valores nulos y tipos de datos
print("📊 INFORMACIÓN GENERAL DE LA MASTER TABLE")
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
print("\n❌ Valores nulos por columna:")
nulos = df.isnull().sum()
print(nulos[nulos > 0])

📊 INFORMACIÓN GENERAL DE LA MASTER TABLE
Filas: 99441 | Columnas: 39

❌ Valores nulos por columna:
order_approved_at                  160
order_delivered_carrier_date      1783
order_delivered_customer_date     2965
payment_value                        1
payment_type                         1
total_freight_value                775
total_price                        775
seller_id                          775
product_id                         775
product_category_name_english     2212
product_weight_g                   791
review_comment_message           58656
delivery_delay_days               2965
actual_delivery_days              2965
approval_time_hours                160
freight_ratio                      775
dias_entrega                      2965
dias_retraso                      2965
dtype: int64


In [ ]:
# ───────────────────────────────────────────────
# KPIs (calculados sobre pedidos entregados con review)
# ───────────────────────────────────────────────
# Base: solo pedidos entregados que tienen review
base = df_delivered.dropna(subset=["review_score"])

kpis = {
    # 1. Tasa de satisfacción (score 4-5)
    "customer_satisfaction_rate": (
        base[base["review_score"] >= 4].shape[0] / base.shape[0] * 100
    ),
    
    # 2. Tasa de insatisfacción (score 1-3)
    "unsatisfied_rate": (
        base[base["review_score"] <= 3].shape[0] / base.shape[0] * 100
    ),
    
    # 3. Score promedio
    "average_review_score": base["review_score"].mean(),
    
    # 4. % de entregas tarde (usando la columna que YA creamos arriba)
    "late_delivery_rate": (
        df_delivered["delivered_on_time"].eq(0).sum() / 
        df_delivered["delivered_on_time"].shape[0] * 100
    ),
    
    # 5. Retraso promedio (solo los que llegaron tarde, en días)
    "average_delay_days": df_delivered.loc[
        df_delivered["delivery_delay_days"] > 0, "delivery_delay_days"
    ].mean(),
    
    # 6. Ingreso en riesgo (pedidos insatisfechos)
    "revenue_at_risk": base[base["review_score"] <= 3]["payment_value"].sum(),
    
    # 7. Valor promedio de orden
    "average_order_value": df_delivered["payment_value"].mean(),
    
    # 8. Satisfacción por estado
    "satisfaction_by_state": base.groupby("customer_state")["review_score"].mean().to_dict(),
    
    # 9. Satisfacción por tipo de pago
    "satisfaction_by_payment": base.groupby("payment_type")["review_score"].mean().to_dict(),
    
    # 10. Satisfacción por cantidad de items
    "satisfaction_by_quantity": base.groupby("order_item_count")["review_score"].mean().to_dict(),
    }